# EDA — Presuntos Suicidios Colombia 2015-2024

Este notebook revisa la **calidad de los datos crudos (bronze)** y deja la evidencia de cada regla de limpieza que aplica el ETL. No modifica nada: todo se lee como texto (`dtype=str`), solo se detecta, se cuantifica y se interpreta.

El análisis descriptivo e inferencial (gráficos, intervalo de confianza y pruebas de hipótesis) está en el documento del trabajo y se hace en R a partir de la capa gold.

**Fuentes**
- `suicidios`: *Presuntos Suicidios. Colombia, 2015 a 2024. Cifras definitivas* (INMLCF), API de Datos Abiertos `f75u-mirk`.
- `poblacion_2005_2017` y `poblacion_2018_2050`: proyecciones de población departamental por área (DANE).

Antes de ejecutarlo: correr `python main.py` una vez (descarga la base a `data/bronze/`) y descargar los dos Excel del DANE en `data/bronze/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))   # para importar src/ desde notebooks/
from src.utils import load_config
from src.extract.extract_data import extract_source
from src.transform.clean_data import quitar_tildes

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

config = load_config()


def clave(texto):
    """Forma normalizada para detectar la misma categoria escrita distinto:
    sin tildes, en minusculas y sin espacios ni signos."""
    if pd.isna(texto):
        return texto
    return "".join(c for c in quitar_tildes(str(texto)).lower() if c.isalnum())


def es_sin_info(serie):
    return serie.map(clave).eq("sininformacion")

# Parte A — Base de presuntos suicidios (INMLCF)

## 1. Carga y tamaño
Cada fila es un caso individual de presunto suicidio. Se esperan **26.558 registros** (la cifra publicada) y 32 columnas, con casos de 2015 a 2024.

In [ ]:
df = extract_source(config["sources"]["suicidios"])
# Solo en memoria (bronze no se toca): mismos nombres de columna que silver,
# sea que el crudo venga de la API (ano_del_hecho) o del CSV descargado (Año del hecho)
df.columns = (df.columns.str.strip().str.lower().map(quitar_tildes)
              .str.replace(r"\W+", "_", regex=True))
df = df.rename(columns={"a_o_del_hecho": "ano_del_hecho"})   # la API entrega el año como a_o_del_hecho
print(df.shape)
df.head()

In [ ]:
df["ano_del_hecho"].value_counts().sort_index()

## 2. Calidad: nulos y "Sin información"
En esta base el dato faltante **no viene vacío**: se registra como la categoría `Sin información` (a veces `Sin Información`). Por eso los nulos del sistema salen en cero y hay que contar esa categoría aparte.

In [ ]:
perfil = pd.DataFrame({
    "nulos": df.isna().sum(),
    "sin_info": df.apply(lambda c: es_sin_info(c).sum()),
    "unicos": df.nunique(),
})
perfil["pct_sin_info"] = (perfil["sin_info"] / len(df) * 100).round(1)
perfil.sort_values("sin_info", ascending=False)

Lo que se ve:
- No hay nulos de sistema en ninguna columna.
- `razon_del_suicidio` tiene 52,9 % de "Sin información", `escolaridad` 16,4 %, `estado_civil` 10,8 %, `escenario_del_hecho` 2,9 %, `zona_del_hecho` 0,6 % y `departamento_del_hecho_dane` 4 casos. Coincide con la Tabla 1 del documento.
- `rango_de_hora_del_hecho_x_3_horas` y `localidad_del_hecho` tienen la mayoría sin dato; no se usan en el análisis.
- `circunstancia_del_hecho_detallada` y `manera_de_muerte` tienen **un solo valor** en toda la base (son constantes): no aportan información.

**Regla:** "Sin información" se conserva como una categoría más (no se imputa ni se eliminan filas) y se unifica su escritura.

## 3. Duplicados

In [ ]:
print("ids repetidos:", df["id"].duplicated().sum())
print("filas idénticas (contando id):", df.duplicated().sum())
print("filas idénticas sin contar id:", df.drop(columns="id").duplicated().sum())

In [ ]:
# Ver el par de filas iguales en todo menos el id
iguales = df[df.drop(columns="id").duplicated(keep=False)]
iguales

El `id` es único. Solo hay **un par de registros** iguales en todas las columnas menos el `id`. Como cada fila es una persona y las variables son categorías amplias (grupo de edad, municipio, mes), dos personas distintas pueden coincidir en todo.

**Regla:** no se elimina ningún registro; la llave es `id`.

## 4. Categorías escritas distinto según el año
Para cada columna se agrupan los valores por su forma normalizada (sin tildes, minúsculas, sin espacios ni signos). Si un mismo grupo tiene más de una escritura, es la misma categoría registrada de formas distintas.

In [ ]:
COLUMNAS_CATEGORICAS = [c for c in df.columns if c != "id" and df[c].nunique() <= 100]

variantes = []
for col in COLUMNAS_CATEGORICAS:
    conteo = df[col].value_counts()
    grupos = conteo.groupby(conteo.index.map(clave))
    for _, g in grupos:
        if len(g) > 1:
            variantes.append({"columna": col, "variantes": " | ".join(g.index), "casos": g.sum()})

pd.DataFrame(variantes)

Para ver en qué años aparece cada escritura (ejemplo con algunas columnas):

In [ ]:
for col in ["mes_del_hecho", "dia_del_hecho", "departamento_del_hecho_dane", "zona_del_hecho",
            "grupo_mayor_menor_de_edad", "razon_del_suicidio", "mecanismo_causal_de_la_lesion_fatal"]:
    con_variantes = [v for fila in variantes if fila["columna"] == col for v in fila["variantes"].split(" | ")]
    if con_variantes:
        sub = df[df[col].isin(con_variantes)]
        display(pd.crosstab(sub[col], sub["ano_del_hecho"]))

Lo que se ve:
- **Mes y día:** en 2021 y 2024 vienen con mayúscula inicial (`Enero`, `Lunes`); los demás años en minúscula.
- **Departamento:** `Bogotá D.C.` (2015-2016) y `Bogotá, D.C.`; `Quindio` (2015) y `Quindío`.
- **Zona:** `Centro poblado(corregimiento…)` y `Centro poblado (corregimiento…)` (2021-2024).
- **Mayor/menor de edad:** en 2015 cambia la redacción (`Mayor de Edad (>18 Años)`).
- **Razón del suicidio:** `Conflicto con pareja o ex-pareja` y `ex pareja` (2022-2023), `Escolares / educativas` con tres escrituras, `Abuso de sustancias / sustancia`, `Accesos a armas`, `Maltrato fisico…` sin tildes, entre otras.
- **Mecanismo causal:** `Cortopunzante` / `Corto punzante`, `Cortocontundente` / `Corto contundente`, tres formas de "explosivo".
- **Estado civil:** `Soltero(a)` / `Soltero (a)`, igual con casado y viudo.

**Regla:** unificar cada columna con `unificar_categorias` (se queda la escritura más frecuente) y mapear a mano los casos que cambian de redacción y no solo de tildes o espacios (explosivo, mayor/menor de edad).

In [ ]:
# Escolaridad: no son solo variantes de escritura, mezcla dos clasificaciones distintas
df["escolaridad"].value_counts()

`escolaridad` combina dos esquemas de clasificación según el año (por ejemplo `Básica primaria` y `Educación básica primaria`, `Preescolar` y `Educación inicial y educación preescolar`). No es una variable del análisis de asociación, así que solo se unifica la escritura y se deja anotado.

## 5. Mecanismo causal de la lesión fatal
El análisis de la paradoja de género recodifica el mecanismo en **violento / no violento** (anexo del documento).

In [ ]:
df["mecanismo_causal_de_la_lesion_fatal"].value_counts()

In [ ]:
VIOLENTO = ["generadoresdeasfixia", "proyectildearmadefuego", "contundente", "cortopunzante", "cortante",
            "cortocontundente", "punzante", "termico", "electrico", "agenteomecanismoexplosivo",
            "agentesymecanismoexplosivo", "mecanismooagenteexplosivo"]
NO_VIOLENTO = ["toxico", "caustico"]

mec = df["mecanismo_causal_de_la_lesion_fatal"].map(clave)
tipo = pd.Series("Por determinar", index=df.index)
tipo[mec.isin(VIOLENTO)] = "Violento"
tipo[mec.isin(NO_VIOLENTO)] = "No violento"

print("Sin clasificar:", sorted(set(mec[tipo.eq("Por determinar")])))
pd.crosstab(df["sexo_de_la_victima"], tipo, margins=True)

Todas las categorías quedan clasificadas, salvo `Por determinar` (35 casos), que se deja aparte. El resultado reproduce la Tabla 4 del documento: 22.288 violentos y 4.235 no violentos.

**Regla:** crear `tipo_mecanismo` (Violento / No violento / Por determinar) en gold.

## 6. Razón del suicidio
Variable con mayor subregistro. Se revisan las frecuencias entre los casos informados para decidir qué categorías se agrupan.

In [ ]:
razon = df["razon_del_suicidio"]
informados = razon[~es_sin_info(razon)]
print("informados:", len(informados), "de", len(razon))

frec = informados.map(clave).value_counts()
pd.DataFrame({"casos": frec, "pct_informados": (frec / len(informados) * 100).round(1)})

De los 12.500 casos con razón informada, nueve categorías concentran casi todo: conflicto de pareja, enfermedad física o mental, enfermedad mental, desamor, económicas, enfermedad física, abuso de sustancias, muerte de un familiar o amigo y jurídicas. Las demás tienen muy pocos casos (bullying, ciberacoso, orientación sexual diversa, acceso a armas, etc.) y producirían frecuencias esperadas menores a 5 en la prueba chi-cuadrado.

**Regla:** crear `razon_agrupada` con esas 9 categorías, `Otras` (las demás) y `Sin información`.

## 7. Departamentos y códigos DANE
La población del DANE se une por **código de departamento y año**, así que los códigos tienen que coincidir.

In [ ]:
pd.crosstab(df["codigo_dane_departamento"], df["departamento_del_hecho_dane"]).pipe(
    lambda t: t.loc[:, (t > 0).any()]
).apply(lambda fila: ", ".join(fila[fila > 0].index), axis=1).to_frame("nombres")

In [ ]:
df["codigo_dane_departamento"].str.len().value_counts()

Lo que se ve:
- 33 departamentos (incluye Bogotá) más `999` = "Sin información" (4 casos).
- Los códigos vienen **sin cero a la izquierda** (`5` Antioquia, `8` Atlántico), mientras que el DANE usa `05` y `08`.
- Un código puede tener dos nombres (Bogotá, Quindío), por eso conviene unir por código y no por nombre.

**Reglas:** rellenar el código a 2 dígitos (`zfill(2)`), tomar el nombre oficial del DANE y agregar la `region` (Amazonía, Andina, Caribe, Orinoquía, Pacífica) para el contraste por regiones. Los 4 casos sin departamento quedan fuera de las tasas departamentales.

# Parte B — Proyecciones de población DANE

Los archivos del DANE traen títulos arriba del encabezado, filas vacías y notas al final. El extractor busca la fila que empieza con `DP` (`header_starts` en `config.yaml`); el archivo unificado `DCD-area-proypoblacion-dep-2005-2050_VP.xlsx` reúne las series 2005-2017 y 2018-2050.

In [ ]:
pob = extract_source(config["sources"]["poblacion_2005_2050"])
print(pob.shape, pob.columns.tolist())
pob.head()

In [ ]:
print("  años:", pob["AÑO"].dropna().unique()[:3], "...", pob["AÑO"].dropna().unique()[-3:])
print("  áreas:", pob["ÁREA GEOGRÁFICA"].dropna().unique())
print("  códigos DP:", pob["DP"].str.len().value_counts().to_dict())

Lo que se ve:
- Cada departamento y año viene en tres filas: `Cabecera`, `Centros Poblados y Rural Disperso` y `Total`; solo se usa `Total`.
- La población viene como texto; se convierte a entero.
- Los códigos `DP` vienen con cero a la izquierda (`05`), mientras que Medicina Legal los trae sin él (`5`).
- Para las tasas por sexo, grupo de edad y municipio se usan además los archivos del DANE por sexo y edad simple y los municipales (ver `config.yaml` e `informe.ipynb`).

Comprobación: con la población nacional del DANE se obtienen las tasas del documento (4,47 en 2015 y 5,73 en 2024).

In [ ]:
def poblacion_total(p, col_pob):
    datos = p[p["ÁREA GEOGRÁFICA"].eq("Total") & p["DP"].str.fullmatch(r"\d{2}", na=False)].copy()
    datos["poblacion"] = pd.to_numeric(datos[col_pob].str.replace(",", "", regex=False))
    return datos.groupby("AÑO")["poblacion"].sum()

poblacion = poblacion_total(pob, "Población")
casos = df["ano_del_hecho"].value_counts()
tasas = pd.DataFrame({"casos": casos, "poblacion": poblacion}).dropna().astype({"casos": int, "poblacion": int})
tasas["tasa_100k"] = (tasas["casos"] / tasas["poblacion"] * 100_000).round(2)
tasas

# Parte C — Análisis exploratorio visual (capa silver)
Hasta aquí se revisó la **calidad** del dato crudo. Esta parte describe **qué dicen los datos**, usando ya la capa **silver** (`data/silver/`), que es la que consume gold. Tampoco modifica nada.

Requiere haber corrido `python main.py` (genera `data/silver/`).

**Nota sobre la edad:** la base **no trae edad exacta** ni fecha exacta del hecho; solo grupos de edad, año, mes y día de la semana. Por eso no hay histograma de edad continua (se usan los grupos quinquenales) ni serie diaria.

In [ ]:
import matplotlib.pyplot as plt
from src.utils import project_path

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})
COLOR, COLOR2 = "#2a6f97", "#c1121f"

silver = project_path(config["paths"]["silver_dir"])
s = pd.read_csv(silver / "suicidios_clean.csv",
                dtype={"codigo_dane_departamento": str, "codigo_dane_municipio": str})
pob = pd.read_csv(silver / "poblacion_2005_2050_clean.csv", dtype={"codigo_dane": str})
print(s.shape, pob.shape)
s.dtypes.to_frame("tipo").T

## C.1 Verificación de la capa silver
Antes de graficar, se comprueba que silver cumple las reglas definidas en el EDA.

In [ ]:
def sin_info(serie):
    return serie.map(clave).eq("sininformacion")

periodo = config["period"]
checks = {
    "id único": s["id"].is_unique,
    "26.558 registros (no se eliminó ninguno)": len(s) == 26558,
    "año dentro del periodo, sin nulos": bool(s["ano_del_hecho"].between(periodo["start_year"], periodo["end_year"]).all()),
    "código depto: 2 dígitos (o 'Sin informacion')":
        bool(s["codigo_dane_departamento"].str.fullmatch(r"\d{2}").sum() + s["codigo_dane_departamento"].eq("Sin informacion").sum() == len(s)),
    "código municipio: 5 dígitos": bool(s["codigo_dane_municipio"].str.fullmatch(r"\d{5}").all()),
    "un nombre por código de departamento": s.groupby("codigo_dane_departamento")["departamento_del_hecho_dane"].nunique().max() == 1,
    "un nombre por código de municipio": s.groupby("codigo_dane_municipio")["municipio_del_hecho_dane"].nunique().max() == 1,
    "sin columnas constantes": not {"manera_de_muerte", "circunstancia_del_hecho_detallada"} & set(s.columns),
    "población sin nulos y entera": bool(pob["poblacion"].notna().all() and str(pob["poblacion"].dtype).startswith("int")),
    "población: una fila por depto y año": not pob.duplicated(["codigo_dane", "ano"]).any(),
}
pd.Series(checks, name="cumple").to_frame()

In [ ]:
# Variantes de escritura que deberían haber desaparecido (comparar con la sección 4)
restantes = []
for col in [c for c in s.columns if c != "id" and s[c].dtype == object and s[c].nunique() <= 100]:
    conteo = s[col].value_counts()
    for _, g in conteo.groupby(conteo.index.map(clave)):
        if len(g) > 1:
            restantes.append({"columna": col, "variantes": " | ".join(g.index), "casos": g.sum()})
pd.DataFrame(restantes) if restantes else print("Sin variantes de escritura en silver")

## C.2 Tendencia anual
Casos por año y **tasa por 100.000 habitantes**. Los casos crudos pueden subir solo porque crece la población; la tasa lo corrige.

In [ ]:
anual = s["ano_del_hecho"].value_counts().sort_index().to_frame("casos")
pop_anual = pob.groupby("ano")["poblacion"].sum()
anual["poblacion"] = pop_anual
anual["tasa_100k"] = (anual["casos"] / anual["poblacion"] * 100_000).round(2)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(anual.index, anual["casos"], marker="o", color=COLOR)
ax[0].set(title="Casos por año", xlabel="Año", ylabel="Casos", xticks=anual.index)
for x, y in anual["casos"].items():
    ax[0].annotate(f"{y:,}".replace(",", "."), (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
ax[1].plot(anual.index, anual["tasa_100k"], marker="o", color=COLOR2)
ax[1].set(title="Tasa por 100.000 habitantes", xlabel="Año", ylabel="Tasa", xticks=anual.index)
for x, y in anual["tasa_100k"].items():
    ax[1].annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
plt.tight_layout()
anual

## C.3 Sexo y mayoría/minoría de edad

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))

sexo = s["sexo_de_la_victima"].value_counts()
ax[0].bar(sexo.index, sexo.values, color=[COLOR, COLOR2][: len(sexo)])
ax[0].set(title="Casos por sexo", ylabel="Casos")
for i, (k, v) in enumerate(sexo.items()):
    ax[0].text(i, v, f"{v:,}\n({v / sexo.sum():.1%})".replace(",", "."), ha="center", va="bottom", fontsize=9)
ax[0].set_ylim(0, sexo.max() * 1.18)

edad = s["grupo_mayor_menor_de_edad"].value_counts()
ax[1].bar(edad.index, edad.values, color=COLOR)
ax[1].set(title="Mayores y menores de edad")
for i, v in enumerate(edad.values):
    ax[1].text(i, v, f"{v:,}\n({v / edad.sum():.1%})".replace(",", "."), ha="center", va="bottom", fontsize=9)
ax[1].set_ylim(0, edad.max() * 1.18)

por_anio = pd.crosstab(s["ano_del_hecho"], s["sexo_de_la_victima"], normalize="index") * 100
por_anio.plot(ax=ax[2], marker="o", color=[COLOR2, COLOR][: por_anio.shape[1]] if list(por_anio.columns) == ["Hombre", "Mujer"] else None)
ax[2].set(title="% de casos por sexo y año", xlabel="Año", ylabel="%", xticks=por_anio.index)
plt.tight_layout()
pd.crosstab(s["sexo_de_la_victima"], s["grupo_mayor_menor_de_edad"], margins=True)

**Paradoja de género.** Los hombres concentran la gran mayoría de los suicidios consumados. Para contrastarla con el mecanismo (violento / no violento, sección 5) se cruza sexo con tipo de mecanismo:

In [ ]:
mec = s["mecanismo_causal_de_la_lesion_fatal"].map(clave)
tipo = pd.Series("Por determinar", index=s.index)
tipo[mec.isin(VIOLENTO)] = "Violento"
tipo[mec.isin(NO_VIOLENTO)] = "No violento"

cruce = pd.crosstab(s["sexo_de_la_victima"], tipo, normalize="index") * 100
cruce[["Violento", "No violento", "Por determinar"]].round(1).plot.bar(stacked=True, figsize=(7, 4),
        color=[COLOR2, COLOR, "#bbbbbb"], rot=0)
plt.title("Tipo de mecanismo por sexo (%)"); plt.ylabel("%"); plt.legend(loc="lower right")
plt.tight_layout()
pd.crosstab(s["sexo_de_la_victima"], tipo, margins=True)

## C.4 Departamentos
Casos absolutos (dependen del tamaño del departamento) frente a **tasa media anual por 100.000 habitantes** (comparable entre departamentos). Los 4 casos sin departamento quedan fuera.

In [ ]:
periodo_anios = range(periodo["start_year"], periodo["end_year"] + 1)
con_depto = s[s["codigo_dane_departamento"].ne("Sin informacion")]
casos_dep = con_depto.groupby(["codigo_dane_departamento", "departamento_del_hecho_dane"]).size().rename("casos").reset_index()
pob_dep = (pob[pob["ano"].isin(periodo_anios)].groupby("codigo_dane")["poblacion"].sum().rename("persona_anio"))
dep = casos_dep.merge(pob_dep, left_on="codigo_dane_departamento", right_index=True, how="left")
print("Departamentos sin población asociada:", dep["persona_anio"].isna().sum())
dep["tasa_100k"] = dep["casos"] / dep["persona_anio"] * 100_000

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top = dep.nlargest(10, "casos").sort_values("casos")
ax[0].barh(top["departamento_del_hecho_dane"], top["casos"], color=COLOR)
ax[0].set(title="Top 10 departamentos: casos (2015-2024)", xlabel="Casos")
top_t = dep.nlargest(10, "tasa_100k").sort_values("tasa_100k")
ax[1].barh(top_t["departamento_del_hecho_dane"], top_t["tasa_100k"], color=COLOR2)
ax[1].axvline(anual["casos"].sum() / anual["poblacion"].loc[list(periodo_anios)].sum() * 100_000,
              color="k", ls="--", lw=1, label="Nacional")
ax[1].set(title="Top 10 departamentos: tasa media anual por 100.000 hab.", xlabel="Tasa"); ax[1].legend()
plt.tight_layout()
dep.sort_values("casos", ascending=False).head(10).round(2)

## C.5 Estacionalidad: mes y día de la semana
Se excluyen los 30 registros "Sin información". Ojo: los meses tienen distinta duración (febrero es más corto), así que se muestra también el promedio de casos **por día del mes**.

In [ ]:
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
DIAS_MES = dict(zip(MESES, [31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]))

mes = s["mes_del_hecho"].str.lower().value_counts().reindex(MESES)
dia = s["dia_del_hecho"].str.lower().value_counts().reindex(DIAS)
print("Meses sin mapear:", set(s["mes_del_hecho"].str.lower()) - set(MESES) - {"sin informacion"},
      "| días sin mapear:", set(s["dia_del_hecho"].str.lower()) - set(DIAS) - {"sin informacion"})

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].bar(range(12), mes.values, color=COLOR); ax[0].set_xticks(range(12)); ax[0].set_xticklabels([m[:3] for m in MESES])
ax[0].set(title="Casos por mes", ylabel="Casos"); ax[0].axhline(mes.mean(), color="k", ls="--", lw=1)
por_dia_mes = mes / pd.Series(DIAS_MES)
ax[1].bar(range(12), por_dia_mes.values, color=COLOR2); ax[1].set_xticks(range(12)); ax[1].set_xticklabels([m[:3] for m in MESES])
ax[1].set(title="Casos por día del mes (corrige duración)"); ax[1].axhline(por_dia_mes.mean(), color="k", ls="--", lw=1)
ax[2].bar(range(7), dia.values, color=COLOR); ax[2].set_xticks(range(7)); ax[2].set_xticklabels([d[:3] for d in DIAS])
ax[2].set(title="Casos por día de la semana"); ax[2].axhline(dia.mean(), color="k", ls="--", lw=1)
plt.tight_layout()
pd.DataFrame({"mes": mes.index, "casos": mes.values, "casos_por_dia": por_dia_mes.round(1).values})

In [ ]:
# ¿El patrón por día de la semana es estable entre años?
mapa = pd.crosstab(s["dia_del_hecho"].str.lower(), s["ano_del_hecho"], normalize="columns").reindex(DIAS) * 100
fig, ax = plt.subplots(figsize=(9, 3.8))
im = ax.imshow(mapa.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(mapa.shape[1])); ax.set_xticklabels(mapa.columns); ax.set_yticks(range(7)); ax.set_yticklabels(DIAS)
for i in range(mapa.shape[0]):
    for j in range(mapa.shape[1]):
        ax.text(j, i, f"{mapa.values[i, j]:.0f}", ha="center", va="center", fontsize=8)
ax.set_title("% de casos de cada año por día de la semana"); ax.grid(False)
plt.colorbar(im); plt.tight_layout()

## C.6 Edad (grupos quinquenales)
Sin edad exacta, la distribución se ve con los grupos quinquenales (equivale a un histograma con barras de 5 años). El grupo `(05 a 09)` tiene muy pocos casos.

In [ ]:
orden = sorted(s["grupo_de_edad_quinquenal"].dropna().unique(), key=lambda g: int("".join(c for c in g.split(" ")[0] if c.isdigit())))
por_grupo = pd.crosstab(s["grupo_de_edad_quinquenal"], s["sexo_de_la_victima"]).reindex(orden)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].bar(range(len(orden)), por_grupo.sum(axis=1).values, color=COLOR, width=0.9)
ax[0].set_xticks(range(len(orden))); ax[0].set_xticklabels(orden, rotation=60, ha="right")
ax[0].set(title="Casos por grupo de edad", ylabel="Casos")
por_grupo.plot.bar(ax=ax[1], color=[COLOR2, COLOR], rot=60, width=0.85)
ax[1].set(title="Casos por grupo de edad y sexo", xlabel="")
plt.tight_layout()
por_grupo.assign(total=por_grupo.sum(axis=1))

## C.7 Resumen visual
- **Tendencia:** los casos pasan de 2.068 (2015) a 3.014 (2024) y la tasa nacional de 4,47 a 5,73 por 100.000 hab. (máximo en 2023: 6,13). 2020 es el único año con descenso claro frente al anterior, y la recuperación posterior es rápida.
- **Sexo:** el 80,4 % de los casos son hombres. El 86,6 % de los hombres usa mecanismos violentos frente al 73,0 % de las mujeres (sección C.3), consistente con la paradoja de género.
- **Edad:** el 10,5 % son menores de edad; los grupos de 20 a 29 años concentran el 26 % de los casos. En las mujeres la distribución se concentra en edades más jóvenes que en los hombres, que mantienen cifras altas hasta edades avanzadas.
- **Territorio:** Antioquia (4.419), Bogotá (3.736) y Valle del Cauca (2.321) lideran en casos absolutos, pero por tasa media anual el orden cambia: Vaupés (13,3), Arauca (10,3), Quindío (8,2), Caldas (7,9) y Risaralda (7,8).
- **Estacionalidad:** por mes no hay un patrón marcado una vez se corrige la duración del mes (entre 70,7 y 74,9 casos por día). Por día de la semana sí: el domingo concentra el 18,1 % y el lunes el 15,7 %, frente a cerca de 13 % de martes a viernes.
- **Silver:** cumple todas las verificaciones de C.1.

# Conclusiones: de los hallazgos a las reglas de limpieza

| # | Hallazgo | Regla en el ETL | Capa |
|---|---|---|---|
| 1 | El dato faltante viene como "Sin información" / "Sin Información", no como nulo | Conservarlo como categoría y unificar su escritura; no imputar ni eliminar filas | Silver |
| 2 | Mes y día con mayúscula en 2021 y 2024; departamento, zona, estado civil, razón y mecanismo con variantes de tildes, espacios y guiones | `unificar_categorias` en cada columna categórica | Silver |
| 3 | Mayor/menor de edad (2015) y "explosivo" cambian de redacción | Mapeo manual a una sola categoría | Silver |
| 4 | `circunstancia_del_hecho_detallada` y `manera_de_muerte` son constantes | Se pueden excluir de gold | Silver |
| 5 | `id` único; un par de registros iguales en el resto de columnas | No eliminar registros | Silver |
| 6 | Códigos de departamento sin cero a la izquierda y nombres con variantes | `zfill(2)` y nombre oficial del DANE; agregar `region` | Silver / Gold |
| 7 | Mecanismo causal con 17 escrituras | `tipo_mecanismo`: Violento / No violento / Por determinar (anexo del documento) | Gold |
| 8 | Razón del suicidio con muchas categorías de pocos casos | `razon_agrupada`: 9 categorías + Otras + Sin información | Gold |
| 9 | DANE: títulos, filas vacías, notas al pie, columna de población con nombre distinto y separador de miles | Leer desde la fila `DP`, quedarse con área `Total`, convertir a número y unir 2015-2017 con 2018-2024 | Silver |
| 10 | La población permite calcular tasas por 100.000 hab. por año y por departamento | Tablas de tasas en gold (nacional y departamental) | Gold |